In [10]:
import pandas as pd
from src.data.loaders import (load_yambda, split_and_reindex)

In [ ]:
config = {
    "yambda-retrieval": {
        "max_seq_len": 100,
        "test_quantile": 0.1,
        "interactions_path": "tmp/yambda-10m.parquet",
        "col_mapping": {
            "uid": "user_id",
            "item_id": "item_id",
            "timestamp": "timestamp",
        },
    },
}

In [5]:
yambda_df = load_yambda(config["yambda-retrieval"]["interactions_path"], config=config["yambda-retrieval"])

In [6]:
yambda_train, yambda_test, yambda_data_description = split_and_reindex(yambda_df, config=config["yambda-retrieval"])

In [7]:
datasets = {
    "yambda": {
        "train": yambda_train,
        "test": yambda_test,
        "desc": yambda_data_description,
    }
}

In [11]:
# статистики по датасетам

stats = []
for name, d in datasets.items():
    train, test, desc = d["train"], d["test"], d["desc"]
    n_train = len(train)
    n_test = len(test)
    n_total = n_train + n_test
    stats.append({
        "dataset": name,
        "n_train": n_train,
        "n_test": n_test,
        "n_total": n_total,
        "train_share": round(n_train / n_total, 3),
        "test_share": round(n_test  / n_total, 3),
        "n_train_users": train[desc["users"]].nunique(),
        "n_test_users": test[desc["users"]].nunique(),
        "n_items": train[desc["items"]].nunique(),
    })

pd.DataFrame(stats)

,dataset,n_train,n_test,n_total,train_share,test_share,n_train_users,n_test_users,n_items
0,yambda,7473676,748537,8222213,0.909,0.091,79059,53927,163448


In [15]:
import torch

from src.training.hstu import (
    HSTUExperimentConfig,
    build_hstu_dataloaders,
    evaluate_hstu,
    train_hstu,
)
from src.models.hstu import HSTUModel


## HSTU Yambda experiment

Запуск HSTU-пайплайна на Yambda

In [16]:
hstu_config = HSTUExperimentConfig(
    max_seq_len=config["yambda-retrieval"]["max_seq_len"],
    train_batch_size=128,
    eval_batch_size=128,
    num_epochs=10,
    learning_rate=1e-3,
    weight_decay=0.0,
    topk=100,
    embedding_dim=64,
    num_blocks=2,
    num_heads=2,
    linear_dim=64,
    attention_dim=64,
    input_dropout_rate=0.2,
    linear_dropout_rate=0.2,
    num_negatives=512,
    sampling_strategy="local",
    softmax_temperature=0.05,
    user_embedding_norm="l2_norm",
    enable_relative_attention_bias = True,
    l2_norm_embeddings=True,
    l2_norm_eps=1e-6,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

hstu_train_loader, hstu_eval_loader, hstu_train_dataset, hstu_eval_dataset, hstu_targets = build_hstu_dataloaders(
    train=yambda_train,
    test=yambda_test,
    max_seq_len=hstu_config.max_seq_len,
    train_batch_size=hstu_config.train_batch_size,
    eval_batch_size=hstu_config.eval_batch_size,
    user_col=yambda_data_description["users"],
    item_col=yambda_data_description["items"],
    time_col=yambda_data_description["timestamp"],
    num_workers=hstu_config.num_workers,
)


len(hstu_train_dataset), len(hstu_eval_dataset), len(hstu_targets)


(121756, 53927, 53927)

In [17]:
hstu_model = HSTUModel(
    num_items=yambda_data_description["n_items"],
    embedding_dim=hstu_config.embedding_dim,
    max_seq_len=hstu_config.max_seq_len,
    num_blocks=hstu_config.num_blocks,
    num_heads=hstu_config.num_heads,
    linear_dim=hstu_config.linear_dim,
    attention_dim=hstu_config.attention_dim,
    num_negatives=hstu_config.num_negatives,
    softmax_temperature=hstu_config.softmax_temperature,
    sampling_strategy=hstu_config.sampling_strategy,
    user_embedding_norm=hstu_config.user_embedding_norm,
    l2_norm_embeddings=hstu_config.l2_norm_embeddings,
    l2_norm_eps=hstu_config.l2_norm_eps,
    item_id_offset=hstu_config.item_id_offset,
    input_dropout_rate=hstu_config.input_dropout_rate,
    linear_dropout_rate=hstu_config.linear_dropout_rate,
    attn_dropout_rate=hstu_config.attn_dropout_rate,
)

hstu_model.to(hstu_config.device)
hstu_optimizer = torch.optim.AdamW(
    hstu_model.parameters(),
    lr=hstu_config.learning_rate,
    weight_decay=hstu_config.weight_decay,
)


In [18]:
params_sum = 0
trainable_sum = 0

for name, module in hstu_model.encoder.named_children():
    params = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    print(f"{name}: total={params:,}, trainable={trainable:,}")

    params_sum += params
    trainable_sum += trainable
print()
print(f"params_sum={params_sum:,}, trainable_sum={trainable_sum:,}")

item_embeddings: total=10,460,736, trainable=10,460,736
pos_embeddings: total=6,400, trainable=6,400
input_dropout: total=0, trainable=0
blocks: total=83,472, trainable=83,472

params_sum=10,550,608, trainable_sum=10,550,608


In [19]:
losses, eval_history = train_hstu(
    model=hstu_model,
    train_loader=hstu_train_loader,
    optimizer=hstu_optimizer,
    num_epochs=hstu_config.num_epochs,
    device=hstu_config.device,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=yambda_data_description["n_items"],
    topk=hstu_config.topk,
    filter_seen=hstu_config.filter_seen,
)
losses

epoch 1/10:   0%|          | 0/951 [00:00<?, ?it/s]

HSTU evaluation:   0%|          | 0/422 [00:00<?, ?it/s]

epoch 1/10: loss=4.8802, coverage=0.0855, hitrate=0.3472, ndcg=0.0316, recall=0.0697


epoch 2/10:   0%|          | 0/951 [00:00<?, ?it/s]

HSTU evaluation:   0%|          | 0/422 [00:00<?, ?it/s]

...

epoch 9/10: loss=2.5094, coverage=0.5779, hitrate=0.4990, ndcg=0.0608, recall=0.1253


epoch 10/10:   0%|          | 0/951 [00:00<?, ?it/s]

HSTU evaluation:   0%|          | 0/422 [00:00<?, ?it/s]

epoch 10/10: loss=2.4780, coverage=0.5942, hitrate=0.4992, ndcg=0.0608, recall=0.1255


[4.8802199143089835,
 3.5085830069240083,
 3.088496361621171,
 2.88985136104307,
 2.75607209000051,
 2.6633724745139715,
 2.5971574720649437,
 2.5473018850813904,
 2.509394324289637,
 2.4779995029280992]

In [20]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=yambda_data_description["n_items"],
    topk=10,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics

HSTU evaluation:   0%|          | 0/422 [00:00<?, ?it/s]

{'hitrate': 0.18243180595990877,
 'recall': 0.03772155538895536,
 'ndcg': 0.03335553624752489,
 'coverage': 0.22319636826391268}

In [21]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=yambda_data_description["n_items"],
    topk=50,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics

HSTU evaluation:   0%|          | 0/422 [00:00<?, ?it/s]

{'hitrate': 0.395738683776216,
 'recall': 0.08343633669724367,
 'ndcg': 0.04806770518668592,
 'coverage': 0.4596140668591846}

In [22]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=yambda_data_description["n_items"],
    topk=100,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics

HSTU evaluation:   0%|          | 0/422 [00:00<?, ?it/s]

{'hitrate': 0.499174810391826,
 'recall': 0.12554879618608492,
 'ndcg': 0.060777557568382455,
 'coverage': 0.594207331995497}

In [23]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=yambda_data_description["n_items"],
    topk=200,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics

HSTU evaluation:   0%|          | 0/422 [00:00<?, ?it/s]

{'hitrate': 0.5994399836816438,
 'recall': 0.1856030194230913,
 'ndcg': 0.07707611246580004,
 'coverage': 0.7356896382947482}

In [ ]:
checkpoint = {
    "epoch": hstu_config.num_epochs,
    "model_state_dict": hstu_model.state_dict(),
    "optimizer_state_dict": hstu_optimizer.state_dict(),
    "loss": losses,
}

torch.save(checkpoint, "yambda_checkpoint.pt")